# Xception Keras -> PyTorch: port fiel + verificacion de paridad

Objetivo: convertir `modelo_xception_fulldatabaseV3100.h5` (Keras/TF) a un modulo PyTorch
identico, para que el gradiente pueda atravesarlo durante el fine-tuning por ciclo del DDPM.

Este cuadernillo **no entrena nada**. Produce tres artefactos en `/kaggle/working`:

1. `xception_regressor_torch.pt` — pesos PyTorch del regresor congelado.
2. `xception_param_weights.json` — R^2 por parametro sobre imagenes reales y los pesos `w_j`.
3. `scaler_check.json` — comparacion de los dos MinMaxScaler (split DDPM vs split Xception).

**Criterio de aceptacion:** la maxima diferencia absoluta entre las salidas de Keras y las de
PyTorch sobre las mismas imagenes debe ser < 1e-3 en el espacio escalado [0,1]. Si no se cumple,
la celda de diagnostico capa-por-capa localiza donde diverge. No tiene sentido montar el
entrenamiento encima de un port que no reproduce al evaluador.

Inputs de Kaggle necesarios:
- `carloscanamejoy/dataset-spines-united-v2`
- `carloscanamejoy/weights-xception-model`

In [1]:
import os, json, math, time, gc
import numpy as np

DATASET_PATH = "/kaggle/input/datasets/carloscanamejoy/dataset-spines-united-v2/dataset_unificado_v2.npz"
XPN_WEIGHTS  = "/kaggle/input/datasets/carloscanamejoy/weights-xception-model/modelo_xception_fulldatabaseV3100.h5"

OUT_DIR = "/kaggle/working/xception_port"
os.makedirs(OUT_DIR, exist_ok=True)
TORCH_CKPT   = f"{OUT_DIR}/xception_regressor_torch.pt"
WEIGHTS_JSON = f"{OUT_DIR}/xception_param_weights.json"
SCALER_JSON  = f"{OUT_DIR}/scaler_check.json"

CROP_TO   = 39
XPN_SIZE  = 224
COND_DIM  = 8
SEED      = 42

PARAM_NAMES = ["T", "Jex2", "Jex3", "Jex4", "Kan1", "KanS", "Hex", "KDM"]

# R^2 reportados en la Tabla 3 del paper (Xception, test set completo).
# Se usan solo como control de sanidad frente a los que midamos aqui.
PAPER_R2 = {"T": 0.96, "Jex2": 0.89, "Jex3": 0.01, "Jex4": 0.00,
            "Kan1": 0.94, "KanS": 0.64, "Hex": 0.96, "KDM": 0.99}

N_PARITY  = 512   # imagenes reales para el test de paridad
XPN_BATCH = 64

for p in (DATASET_PATH, XPN_WEIGHTS):
    if not os.path.exists(p):
        raise FileNotFoundError(f"No existe: {p}. Revisa los Add Input de Kaggle.")
print("Rutas OK")
print(f"Salidas en: {OUT_DIR}")

Rutas OK
Salidas en: /kaggle/working/xception_port


## 1. Comprobacion de scalers

El DDPM se entreno con un `MinMaxScaler` ajustado sobre su propio split (permutacion con
`SEED=42`, luego 70/15/15), mientras que `xception-eval-ddpm` construye el condicionamiento con
un scaler ajustado sobre el split del Xception (85/15 y luego 0.1765). Si los `data_min_` /
`data_max_` coinciden, ambos espacios son el mismo [0,1] y el termino de ciclo se calcula
directamente ahi, sin `inverse_transform`. Si no coinciden, los R^2 medidos hasta ahora estan
contaminados porque el DDPM recibio condicionamiento fuera de su espacio de entrenamiento.

Nota: `train_test_split(X, y, random_state=42)` produce exactamente la misma particion que
`train_test_split(arange(N), random_state=42)`, porque el barajado depende solo de `n_samples`
y de la semilla. Por eso aqui se parten indices y no arreglos, lo que ahorra ~1 GB de RAM.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

_d = np.load(DATASET_PATH)
params_all = _d["params"].astype(np.float32)
N = len(params_all)
print(f"Dataset: {N:,} muestras  |  params: {params_all.shape}")

# --- Split del DDPM (make_split del cuadernillo ddpm) ---
rng = np.random.RandomState(SEED)
sub_idx = rng.choice(N, size=int(N * 1.0), replace=False)
params_s = params_all[sub_idx]
idx_all = np.arange(len(sub_idx))
idx_tr_d, idx_tmp_d = train_test_split(idx_all, test_size=0.30, random_state=SEED)
idx_va_d, idx_te_d = train_test_split(idx_tmp_d, test_size=0.50, random_state=SEED)
sc_ddpm = MinMaxScaler().fit(params_s[idx_tr_d])

# --- Split del Xception (xception-eval-ddpm / seccion 4.2 del paper) ---
idx = np.arange(N)
idx_trv_x, idx_te_x = train_test_split(idx, test_size=0.15, random_state=SEED)
idx_tr_x, idx_va_x = train_test_split(idx_trv_x, test_size=0.1765, random_state=SEED)
sc_xpn = MinMaxScaler().fit(params_all[idx_tr_x])

print(f"\nDDPM      train={len(idx_tr_d):,}  val={len(idx_va_d):,}  test={len(idx_te_d):,}")
print(f"Xception  train={len(idx_tr_x):,}  val={len(idx_va_x):,}  test={len(idx_te_x):,}")

d_min = np.abs(sc_ddpm.data_min_ - sc_xpn.data_min_)
d_max = np.abs(sc_ddpm.data_max_ - sc_xpn.data_max_)
rng_x = np.maximum(sc_xpn.data_max_ - sc_xpn.data_min_, 1e-12)
rel = np.maximum(d_min, d_max) / rng_x

print("\n  param     min(ddpm)    min(xpn)     max(ddpm)    max(xpn)     err.rel")
for j, pn in enumerate(PARAM_NAMES):
    print(f"  {pn:8s} {sc_ddpm.data_min_[j]:11.6f} {sc_xpn.data_min_[j]:11.6f} "
          f"{sc_ddpm.data_max_[j]:11.6f} {sc_xpn.data_max_[j]:11.6f} {rel[j]:11.2e}")

SCALERS_MATCH = bool(np.all(rel < 1e-6))
print(f"\nScalers equivalentes: {SCALERS_MATCH}")
if SCALERS_MATCH:
    print("  -> El condicionamiento del DDPM y el target del Xception viven en el mismo [0,1].")
    print("     El termino de ciclo se calcula directamente en ese espacio.")
else:
    print("  -> NO coinciden. Los R2 medidos hasta ahora estan contaminados:")
    print("     el DDPM recibio condicionamiento fuera de su espacio de entrenamiento.")
    print("     Habra que mapear cond -> fisico -> escala Xception en el termino de ciclo.")

json.dump({"scalers_match": SCALERS_MATCH,
           "param_names": PARAM_NAMES,
           "ddpm_min": sc_ddpm.data_min_.tolist(), "ddpm_max": sc_ddpm.data_max_.tolist(),
           "xpn_min": sc_xpn.data_min_.tolist(),   "xpn_max": sc_xpn.data_max_.tolist(),
           "rel_err": rel.tolist()}, open(SCALER_JSON, "w"), indent=2)
print(f"\nGuardado: {SCALER_JSON}")

Dataset: 169,671 muestras  |  params: (169671, 8)

DDPM      train=118,769  val=25,451  test=25,451
Xception  train=118,765  val=25,455  test=25,451

  param     min(ddpm)    min(xpn)     max(ddpm)    max(xpn)     err.rel
  T           0.000000    0.000000   20.000000   20.000000    0.00e+00
  Jex2       -0.286000   -0.286000    0.659000    0.659000    0.00e+00
  Jex3       -0.290000   -0.290000    0.290000    0.290000    0.00e+00
  Jex4       -0.234000   -0.234000    0.235000    0.235000    0.00e+00
  Kan1        0.000000    0.000000    0.599700    0.599700    0.00e+00
  KanS        0.000000    0.000000    0.200000    0.200000    0.00e+00
  Hex         0.000000    0.000000    1.198800    1.198800    0.00e+00
  KDM         0.000000    0.000000    1.200000    1.200000    0.00e+00

Scalers equivalentes: True
  -> El condicionamiento del DDPM y el target del Xception viven en el mismo [0,1].
     El termino de ciclo se calcula directamente en ese espacio.

Guardado: /kaggle/working/xcepti

## 2. Cargar el modelo Keras

Se reconstruye exactamente la arquitectura del cuadernillo de evaluacion (Xception sin top +
la cabeza BN -> Drop(0.4) -> Dense(256, relu, l2) -> BN -> Drop(0.3) -> Dense(8), que es la
misma descrita en la Tabla 2 del paper) y se cargan los pesos del `.h5`.

Si hay dos GPUs, TensorFlow se restringe a la primera y PyTorch usa la segunda, para que no
compitan por VRAM. Con una sola, comparten con `memory_growth` activado.

In [3]:
import tensorflow as tf
from tensorflow.keras import Model, regularizers
from tensorflow.keras.layers import (Input, GlobalAveragePooling2D, Dense,
                                     Dropout, BatchNormalization)
from tensorflow.keras.applications import Xception

_gpus = tf.config.list_physical_devices("GPU")
for g in _gpus:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass

TORCH_DEVICE = "cuda:0"
if len(_gpus) > 1:
    try:
        tf.config.set_visible_devices(_gpus[0], "GPU")
        TORCH_DEVICE = "cuda:1"
        print("2+ GPUs: TF en GPU0, PyTorch en cuda:1")
    except Exception as e:
        print(f"No se pudo separar GPUs ({e}); se comparte cuda:0")
print(f"TF {tf.__version__}  GPUs visibles para TF: {tf.config.list_physical_devices('GPU')}")

input_layer = Input(shape=(XPN_SIZE, XPN_SIZE, 3))
base_model = Xception(weights=None, include_top=False, input_tensor=input_layer)
x = GlobalAveragePooling2D()(base_model.output)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
x = Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
outputs = Dense(8, activation="linear")(x)
xpn_model = Model(inputs=input_layer, outputs=outputs)
xpn_model.load_weights(XPN_WEIGHTS)
print(f"Xception Keras cargado: {xpn_model.count_params():,} params")

@tf.function
def keras_predict(imgs_raw_4d):
    # Replica exacta del preproceso de xception-eval-ddpm: resize bilineal a 224
    # y grayscale -> RGB. Sin preprocess_input, igual que en el flujo original.
    z = tf.image.resize(imgs_raw_4d, (XPN_SIZE, XPN_SIZE))
    z = tf.image.grayscale_to_rgb(z)
    return xpn_model(z, training=False)

2+ GPUs: TF en GPU0, PyTorch en cuda:1
TF 2.20.0  GPUs visibles para TF: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


I0000 00:00:1788973468.330175      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5


Xception Keras cargado: 21,397,296 params


## 3. Arquitectura PyTorch equivalente

Dos detalles hacen la diferencia entre un port correcto y uno que casi funciona:

**Padding de los max-pool.** Keras usa `padding='same'`, que para kernel 3 / stride 2 reparte el
relleno de forma asimetrica cuando el tamano de entrada es par. En la ruta de Xception a 224 los
pools operan sobre 109 -> 55, 55 -> 28, 28 -> 14 y 14 -> 7; los dos ultimos necesitan relleno
total 1, que TF pone entero a la derecha/abajo. Un `MaxPool2d(padding=1)` de PyTorch rellenaria
ambos lados y desalinearia el mapa. Por eso se calcula el padding al estilo TF y se rellena con
`-inf`, que es como TF ignora las celdas rellenadas en un maximo.

**Epsilon de BatchNorm.** El valor por defecto difiere entre frameworks (Keras 1e-3, PyTorch 1e-5).
No se asume ninguno: se lee de cada capa Keras durante la transferencia.

Las convoluciones separables se descomponen en depthwise (`groups=in_ch`) + pointwise 1x1, que es
literalmente lo que hace `SeparableConv2D` (Eq. 11 del paper).

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F


def tf_same_maxpool(x, k=3, s=2):
    # Replica MaxPooling2D(k, strides=s, padding='same') de TF, incluido el
    # reparto asimetrico del relleno. Se rellena con -inf para que las celdas
    # artificiales nunca ganen el maximo.
    ih, iw = x.shape[-2], x.shape[-1]
    oh, ow = math.ceil(ih / s), math.ceil(iw / s)
    ph = max((oh - 1) * s + k - ih, 0)
    pw = max((ow - 1) * s + k - iw, 0)
    if ph or pw:
        x = F.pad(x, (pw // 2, pw - pw // 2, ph // 2, ph - ph // 2), value=float("-inf"))
    return F.max_pool2d(x, k, s)


class SepConv(nn.Module):
    # SeparableConv2D(out, 3, padding='same', use_bias=False)
    def __init__(self, cin, cout):
        super().__init__()
        self.depthwise = nn.Conv2d(cin, cin, 3, padding=1, groups=cin, bias=False)
        self.pointwise = nn.Conv2d(cin, cout, 1, bias=False)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))


class XceptionRegressor(nn.Module):
    # Xception (Chollet 2017) + la cabeza de regresion de 8 salidas.
    def __init__(self, n_out=8):
        super().__init__()
        # --- entry flow ---
        self.conv1 = nn.Conv2d(3, 32, 3, stride=2, bias=False);  self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, bias=False);           self.bn2 = nn.BatchNorm2d(64)

        self.res1 = nn.Conv2d(64, 128, 1, stride=2, bias=False); self.res1_bn = nn.BatchNorm2d(128)
        self.b2_sc1 = SepConv(64, 128);   self.b2_bn1 = nn.BatchNorm2d(128)
        self.b2_sc2 = SepConv(128, 128);  self.b2_bn2 = nn.BatchNorm2d(128)

        self.res2 = nn.Conv2d(128, 256, 1, stride=2, bias=False); self.res2_bn = nn.BatchNorm2d(256)
        self.b3_sc1 = SepConv(128, 256);  self.b3_bn1 = nn.BatchNorm2d(256)
        self.b3_sc2 = SepConv(256, 256);  self.b3_bn2 = nn.BatchNorm2d(256)

        self.res3 = nn.Conv2d(256, 728, 1, stride=2, bias=False); self.res3_bn = nn.BatchNorm2d(728)
        self.b4_sc1 = SepConv(256, 728);  self.b4_bn1 = nn.BatchNorm2d(728)
        self.b4_sc2 = SepConv(728, 728);  self.b4_bn2 = nn.BatchNorm2d(728)

        # --- middle flow: 8 bloques identicos ---
        self.mid_sc = nn.ModuleList()
        self.mid_bn = nn.ModuleList()
        for _ in range(8):
            self.mid_sc.append(nn.ModuleList([SepConv(728, 728) for _ in range(3)]))
            self.mid_bn.append(nn.ModuleList([nn.BatchNorm2d(728) for _ in range(3)]))

        # --- exit flow ---
        self.res4 = nn.Conv2d(728, 1024, 1, stride=2, bias=False); self.res4_bn = nn.BatchNorm2d(1024)
        self.b13_sc1 = SepConv(728, 728);   self.b13_bn1 = nn.BatchNorm2d(728)
        self.b13_sc2 = SepConv(728, 1024);  self.b13_bn2 = nn.BatchNorm2d(1024)
        self.b14_sc1 = SepConv(1024, 1536); self.b14_bn1 = nn.BatchNorm2d(1536)
        self.b14_sc2 = SepConv(1536, 2048); self.b14_bn2 = nn.BatchNorm2d(2048)

        # --- cabeza de regresion ---
        self.head_bn1 = nn.BatchNorm1d(2048)
        self.head_fc1 = nn.Linear(2048, 256)
        self.head_bn2 = nn.BatchNorm1d(256)
        self.head_out = nn.Linear(256, n_out)

    def features(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))

        # block2: sin activacion antes de la primera separable
        res = self.res1_bn(self.res1(x))
        x = self.b2_bn1(self.b2_sc1(x))
        x = self.b2_bn2(self.b2_sc2(F.relu(x)))
        x = tf_same_maxpool(x) + res

        for res_c, res_b, sc1, bn1, sc2, bn2 in [
            (self.res2, self.res2_bn, self.b3_sc1, self.b3_bn1, self.b3_sc2, self.b3_bn2),
            (self.res3, self.res3_bn, self.b4_sc1, self.b4_bn1, self.b4_sc2, self.b4_bn2),
        ]:
            res = res_b(res_c(x))
            x = bn1(sc1(F.relu(x)))
            x = bn2(sc2(F.relu(x)))
            x = tf_same_maxpool(x) + res

        for scs, bns in zip(self.mid_sc, self.mid_bn):
            res = x
            for sc, bn in zip(scs, bns):
                x = bn(sc(F.relu(x)))
            x = x + res

        res = self.res4_bn(self.res4(x))
        x = self.b13_bn1(self.b13_sc1(F.relu(x)))
        x = self.b13_bn2(self.b13_sc2(F.relu(x)))
        x = tf_same_maxpool(x) + res

        x = F.relu(self.b14_bn1(self.b14_sc1(x)))
        x = F.relu(self.b14_bn2(self.b14_sc2(x)))
        return x

    def forward(self, x):
        x = self.features(x).mean(dim=(2, 3))   # GlobalAveragePooling2D
        x = self.head_bn1(x)                    # Dropout(0.4) es identidad en eval
        x = F.relu(self.head_fc1(x))
        x = self.head_bn2(x)                    # Dropout(0.3) es identidad en eval
        return self.head_out(x)


def preprocess_torch(x_raw_bchw):
    # Equivalente diferenciable de tf.image.resize + grayscale_to_rgb.
    # tf.image.resize usa half-pixel centers sin antialias, que corresponde a
    # align_corners=False en PyTorch.
    z = F.interpolate(x_raw_bchw, size=(XPN_SIZE, XPN_SIZE),
                      mode="bilinear", align_corners=False)
    return z.repeat(1, 3, 1, 1)


torch_model = XceptionRegressor(n_out=COND_DIM)
n_par = sum(p.numel() for p in torch_model.parameters())
print(f"PyTorch XceptionRegressor: {n_par:,} params entrenables + buffers de BN")

PyTorch XceptionRegressor: 21,338,160 params entrenables + buffers de BN


## 4. Transferencia de pesos

Las capas con nombre estable (`block1_conv1`, `block2_sepconv1`, ...) se emparejan por nombre.
Las cuatro convoluciones 1x1 de los atajos residuales no llevan nombre explicito en
`keras.applications.xception`, asi que reciben nombres automaticos (`conv2d`, `conv2d_1`, ...)
que dependen del contador global de Keras y no son fiables. Se localizan por tipo y forma, y se
verifica que sus anchos sean exactamente `[128, 256, 728, 1024]` en orden de grafo.

Conversiones de layout:
- `Conv2D`: `(kh, kw, in, out)` -> `(out, in, kh, kw)`
- depthwise de `SeparableConv2D`: `(kh, kw, in, 1)` -> `(in, 1, kh, kw)`
- pointwise: `(1, 1, in, out)` -> `(out, in, 1, 1)`
- `Dense`: `(in, out)` -> `(out, in)`
- `BatchNormalization`: `[gamma, beta, mean, var]` -> `weight, bias, running_mean, running_var`

In [5]:
def _cls(l):
    return l.__class__.__name__


def set_conv(t_mod, k_layer):
    w = k_layer.get_weights()
    assert len(w) == 1, f"{k_layer.name}: se esperaba solo kernel (use_bias=False)"
    t_mod.weight.data.copy_(torch.from_numpy(w[0].transpose(3, 2, 0, 1).copy()))


def set_sep(t_mod, k_layer):
    dw, pw = k_layer.get_weights()
    t_mod.depthwise.weight.data.copy_(torch.from_numpy(dw.transpose(2, 3, 0, 1).copy()))
    t_mod.pointwise.weight.data.copy_(torch.from_numpy(pw.transpose(3, 2, 0, 1).copy()))


def set_bn(t_mod, k_layer):
    g, b, m, v = k_layer.get_weights()
    t_mod.weight.data.copy_(torch.from_numpy(g.copy()))
    t_mod.bias.data.copy_(torch.from_numpy(b.copy()))
    t_mod.running_mean.data.copy_(torch.from_numpy(m.copy()))
    t_mod.running_var.data.copy_(torch.from_numpy(v.copy()))
    t_mod.eps = float(k_layer.epsilon)   # Keras 1e-3 vs PyTorch 1e-5: se toma el de Keras


def set_dense(t_mod, k_layer):
    w, b = k_layer.get_weights()
    t_mod.weight.data.copy_(torch.from_numpy(w.T.copy()))
    t_mod.bias.data.copy_(torch.from_numpy(b.copy()))


named = {l.name: l for l in base_model.layers}

# Atajos residuales: Conv2D 1x1 sin nombre 'block*', en orden de grafo.
res_convs = [l for l in base_model.layers
             if _cls(l) == "Conv2D" and not l.name.startswith("block")]
res_bns = [l for l in base_model.layers
           if _cls(l) == "BatchNormalization" and not l.name.startswith("block")]
assert len(res_convs) == 4, f"Se esperaban 4 conv residuales, hay {len(res_convs)}: {[l.name for l in res_convs]}"
assert len(res_bns) == 4, f"Se esperaban 4 BN residuales, hay {len(res_bns)}"
assert [l.filters for l in res_convs] == [128, 256, 728, 1024], \
    f"Orden inesperado de atajos: {[l.filters for l in res_convs]}"
print(f"Atajos residuales localizados: {[l.name for l in res_convs]}")

# --- entry flow ---
set_conv(torch_model.conv1, named["block1_conv1"]); set_bn(torch_model.bn1, named["block1_conv1_bn"])
set_conv(torch_model.conv2, named["block1_conv2"]); set_bn(torch_model.bn2, named["block1_conv2_bn"])

for i, (rc, rb) in enumerate(zip(res_convs, res_bns)):
    t_c = [torch_model.res1, torch_model.res2, torch_model.res3, torch_model.res4][i]
    t_b = [torch_model.res1_bn, torch_model.res2_bn, torch_model.res3_bn, torch_model.res4_bn][i]
    set_conv(t_c, rc); set_bn(t_b, rb)

for blk, (sc1, bn1, sc2, bn2) in {
    2: (torch_model.b2_sc1, torch_model.b2_bn1, torch_model.b2_sc2, torch_model.b2_bn2),
    3: (torch_model.b3_sc1, torch_model.b3_bn1, torch_model.b3_sc2, torch_model.b3_bn2),
    4: (torch_model.b4_sc1, torch_model.b4_bn1, torch_model.b4_sc2, torch_model.b4_bn2),
}.items():
    set_sep(sc1, named[f"block{blk}_sepconv1"]); set_bn(bn1, named[f"block{blk}_sepconv1_bn"])
    set_sep(sc2, named[f"block{blk}_sepconv2"]); set_bn(bn2, named[f"block{blk}_sepconv2_bn"])

# --- middle flow: bloques 5..12 ---
for i, blk in enumerate(range(5, 13)):
    for j in range(3):
        set_sep(torch_model.mid_sc[i][j], named[f"block{blk}_sepconv{j+1}"])
        set_bn(torch_model.mid_bn[i][j], named[f"block{blk}_sepconv{j+1}_bn"])

# --- exit flow ---
set_sep(torch_model.b13_sc1, named["block13_sepconv1"]); set_bn(torch_model.b13_bn1, named["block13_sepconv1_bn"])
set_sep(torch_model.b13_sc2, named["block13_sepconv2"]); set_bn(torch_model.b13_bn2, named["block13_sepconv2_bn"])
set_sep(torch_model.b14_sc1, named["block14_sepconv1"]); set_bn(torch_model.b14_bn1, named["block14_sepconv1_bn"])
set_sep(torch_model.b14_sc2, named["block14_sepconv2"]); set_bn(torch_model.b14_bn2, named["block14_sepconv2_bn"])

# --- cabeza (capas que no pertenecen al backbone) ---
base_names = {l.name for l in base_model.layers}
head_layers = [l for l in xpn_model.layers if l.name not in base_names]
head_bns = [l for l in head_layers if _cls(l) == "BatchNormalization"]
head_dns = [l for l in head_layers if _cls(l) == "Dense"]
assert len(head_bns) == 2 and len(head_dns) == 2, \
    f"Cabeza inesperada: {[(l.name, _cls(l)) for l in head_layers]}"
set_bn(torch_model.head_bn1, head_bns[0]); set_dense(torch_model.head_fc1, head_dns[0])
set_bn(torch_model.head_bn2, head_bns[1]); set_dense(torch_model.head_out, head_dns[1])

torch_model.eval().to(TORCH_DEVICE)
for p in torch_model.parameters():
    p.requires_grad_(False)
print(f"Transferencia completa. Modelo en {TORCH_DEVICE}, congelado.")

Atajos residuales localizados: ['conv2d', 'conv2d_1', 'conv2d_2', 'conv2d_3']
Transferencia completa. Modelo en cuda:1, congelado.


## 5. Test de paridad

Se comparan las salidas de Keras y de PyTorch sobre las mismas imagenes reales del test del
Xception, con el preproceso completo (crudo 39x39 -> resize 224 -> RGB) en cada framework.

In [6]:
imgs_all = np.load(DATASET_PATH)["img"].astype(np.float32)
if imgs_all.ndim == 3:
    imgs_all = imgs_all[..., np.newaxis]
DATA_MIN, DATA_MAX = float(imgs_all.min()), float(imgs_all.max())
print(f"Rango crudo del dataset: [{DATA_MIN:.4f}, {DATA_MAX:.4f}]")

test_imgs = imgs_all[idx_te_x]          # (n_test, 39, 39, 1) crudo
test_params = params_all[idx_te_x]
del imgs_all; gc.collect()
print(f"Test del Xception: {len(test_imgs):,} imagenes")

par_imgs = test_imgs[:N_PARITY]

# --- Keras ---
keras_out = []
for i0 in range(0, len(par_imgs), XPN_BATCH):
    keras_out.append(keras_predict(tf.constant(par_imgs[i0:i0 + XPN_BATCH])).numpy())
keras_out = np.concatenate(keras_out, 0)

# --- PyTorch ---
torch_out = []
with torch.no_grad():
    for i0 in range(0, len(par_imgs), XPN_BATCH):
        b = torch.from_numpy(par_imgs[i0:i0 + XPN_BATCH]).permute(0, 3, 1, 2).to(TORCH_DEVICE)
        torch_out.append(torch_model(preprocess_torch(b)).float().cpu().numpy())
torch_out = np.concatenate(torch_out, 0)

diff = np.abs(keras_out - torch_out)
print(f"\nParidad sobre {len(par_imgs)} imagenes reales (espacio escalado [0,1]):")
print(f"  max |Keras - Torch|   = {diff.max():.3e}")
print(f"  media |Keras - Torch| = {diff.mean():.3e}")
print("\n  param     max.dif      corr")
for j, pn in enumerate(PARAM_NAMES):
    c = np.corrcoef(keras_out[:, j], torch_out[:, j])[0, 1]
    print(f"  {pn:8s} {diff[:, j].max():.3e}   {c:.8f}")

PARITY_OK = diff.max() < 1e-3
print(f"\nPARIDAD {'OK' if PARITY_OK else 'FALLIDA'} (umbral 1e-3)")
if not PARITY_OK:
    print("  -> Ejecuta la celda de diagnostico capa-por-capa para localizar la divergencia.")

Rango crudo del dataset: [-1.0000, 1.0000]
Test del Xception: 25,451 imagenes

Paridad sobre 512 imagenes reales (espacio escalado [0,1]):
  max |Keras - Torch|   = 6.974e-06
  media |Keras - Torch| = 3.488e-07

  param     max.dif      corr
  T        2.444e-06   1.00000000
  Jex2     4.232e-06   1.00000000
  Jex3     4.500e-06   1.00000000
  Jex4     4.590e-06   1.00000000
  Kan1     6.974e-06   1.00000000
  KanS     4.113e-06   1.00000000
  Hex      6.735e-06   1.00000000
  KDM      3.099e-06   1.00000000

PARIDAD OK (umbral 1e-3)


### Diagnostico capa por capa (solo si la paridad falla)

Compara activaciones intermedias en puntos de corte del backbone. La primera capa donde el error
salte varios ordenes de magnitud senala el problema: casi siempre es el padding de un max-pool
o el epsilon de un BatchNorm.

In [7]:
def debug_layerwise(n=32):
    # OJO: en Keras la salida de block*_pool es ANTES de sumar el atajo residual,
    # asi que aqui se devuelve el pool sin sumar.
    probes = ["block1_conv2_act", "block2_pool", "block3_pool",
              "block4_pool", "block13_pool", "block14_sepconv2_act"]
    xb = par_imgs[:n]
    xt = preprocess_torch(torch.from_numpy(xb).permute(0, 3, 1, 2).to(TORCH_DEVICE))

    def torch_upto(tag):
        m = torch_model
        with torch.no_grad():
            x = F.relu(m.bn1(m.conv1(xt)))
            x = F.relu(m.bn2(m.conv2(x)))
            if tag == "block1_conv2_act":
                return x
            res = m.res1_bn(m.res1(x))
            x = m.b2_bn1(m.b2_sc1(x)); x = m.b2_bn2(m.b2_sc2(F.relu(x)))
            x = tf_same_maxpool(x)
            if tag == "block2_pool":
                return x
            x = x + res
            for rc, rb, s1, n1, s2, n2, t in [
                (m.res2, m.res2_bn, m.b3_sc1, m.b3_bn1, m.b3_sc2, m.b3_bn2, "block3_pool"),
                (m.res3, m.res3_bn, m.b4_sc1, m.b4_bn1, m.b4_sc2, m.b4_bn2, "block4_pool"),
            ]:
                res = rb(rc(x))
                x = n1(s1(F.relu(x))); x = n2(s2(F.relu(x)))
                x = tf_same_maxpool(x)
                if tag == t:
                    return x
                x = x + res
            for scs, bns in zip(m.mid_sc, m.mid_bn):
                r = x
                for sc, bn in zip(scs, bns):
                    x = bn(sc(F.relu(x)))
                x = x + r
            res = m.res4_bn(m.res4(x))
            x = m.b13_bn1(m.b13_sc1(F.relu(x))); x = m.b13_bn2(m.b13_sc2(F.relu(x)))
            x = tf_same_maxpool(x)
            if tag == "block13_pool":
                return x
            x = x + res
            x = F.relu(m.b14_bn1(m.b14_sc1(x)))
            x = F.relu(m.b14_bn2(m.b14_sc2(x)))
            return x

    print("  punto de corte          shape                 max.dif")
    for tag in probes:
        try:
            sub = Model(inputs=xpn_model.input, outputs=base_model.get_layer(tag).output)
        except Exception:
            cands = [l.name for l in base_model.layers if tag.split("_")[0] in l.name]
            print(f"  {tag:22s} no encontrado en Keras. Candidatos: {cands[:6]}")
            continue
        z = tf.image.grayscale_to_rgb(tf.image.resize(tf.constant(xb), (XPN_SIZE, XPN_SIZE)))
        k_act = sub(z, training=False).numpy().transpose(0, 3, 1, 2)
        t_act = torch_upto(tag).float().cpu().numpy()
        d = np.abs(k_act - t_act).max()
        print(f"  {tag:22s} {str(t_act.shape):20s}  {d:.3e}")

if not PARITY_OK:
    debug_layerwise()
else:
    print("Paridad OK; diagnostico no necesario. Llama a debug_layerwise() si quieres verlo igual.")

Paridad OK; diagnostico no necesario. Llama a debug_layerwise() si quieres verlo igual.


## 6. R^2 sobre imagenes reales y pesos por parametro

Se recalculan los R^2 con el modelo portado sobre el test del Xception (el split de este dataset,
no el del paper) y se derivan los pesos de confianza `w_j = max(0, R^2_j) / sum_j max(0, R^2_j)`.

La Tabla 3 del paper sirve de control: se espera el mismo patron, con J3 y J4 en cero. El paper
establece que su inidentificabilidad es una propiedad fisica de la proyeccion s_z, no un fallo
del regresor, asi que un peso nulo para ellos es lo correcto, no una concesion.

In [8]:
from sklearn.metrics import r2_score, mean_absolute_error

pred_scaled = np.empty((len(test_imgs), COND_DIM), dtype=np.float32)
t0 = time.time()
with torch.no_grad():
    for i0 in range(0, len(test_imgs), XPN_BATCH):
        i1 = min(i0 + XPN_BATCH, len(test_imgs))
        b = torch.from_numpy(test_imgs[i0:i1]).permute(0, 3, 1, 2).to(TORCH_DEVICE)
        pred_scaled[i0:i1] = torch_model(preprocess_torch(b)).float().cpu().numpy()
print(f"Inferencia sobre {len(test_imgs):,} imagenes en {time.time()-t0:.1f}s")

pred = sc_xpn.inverse_transform(pred_scaled)

r2s, maes = [], []
print("\n  param      R2(aqui)   R2(paper)    MAE")
for j, pn in enumerate(PARAM_NAMES):
    r2 = r2_score(test_params[:, j], pred[:, j])
    mae = mean_absolute_error(test_params[:, j], pred[:, j])
    r2s.append(float(r2)); maes.append(float(mae))
    print(f"  {pn:8s} {r2:+9.3f}  {PAPER_R2[pn]:+9.2f}  {mae:9.4f}")

r2_arr = np.array(r2s)
w_raw = np.maximum(r2_arr, 0.0)
w = w_raw / max(w_raw.sum(), 1e-12)
print("\nPesos de confianza w_j (suma 1):")
for j, pn in enumerate(PARAM_NAMES):
    print(f"  {pn:8s} w = {w[j]:.4f}")

# Escalas por parametro: adimensionalizan el error del termino de ciclo.
s_j = params_all[idx_tr_x].std(axis=0)
print("\nDesviaciones estandar s_j (train del Xception):")
for j, pn in enumerate(PARAM_NAMES):
    print(f"  {pn:8s} s = {s_j[j]:.5f}")

json.dump({"param_names": PARAM_NAMES,
           "r2_measured": r2s, "r2_paper": [PAPER_R2[p] for p in PARAM_NAMES],
           "mae": maes, "w": w.tolist(), "param_std": s_j.tolist(),
           "n_test": int(len(test_imgs))},
          open(WEIGHTS_JSON, "w"), indent=2)
print(f"\nGuardado: {WEIGHTS_JSON}")

Inferencia sobre 25,451 imagenes en 113.3s

  param      R2(aqui)   R2(paper)    MAE
  T           +0.946      +0.96     0.4082
  Jex2        +0.878      +0.89     0.0176
  Jex3        -0.083      +0.01     0.0284
  Jex4        -0.174      +0.00     0.0237
  Kan1        +0.617      +0.94     0.0464
  KanS        +0.670      +0.64     0.0257
  Hex         +0.902      +0.96     0.0188
  KDM         +0.988      +0.99     0.0234

Pesos de confianza w_j (suma 1):
  T        w = 0.1891
  Jex2     w = 0.1755
  Jex3     w = 0.0000
  Jex4     w = 0.0000
  Kan1     w = 0.1234
  KanS     w = 0.1339
  Hex      w = 0.1804
  KDM      w = 0.1976

Desviaciones estandar s_j (train del Xception):
  T        s = 3.15462
  Jex2     s = 0.14065
  Jex3     s = 0.07310
  Jex4     s = 0.05866
  Kan1     s = 0.10836
  KanS     s = 0.06252
  Hex      s = 0.17143
  KDM      s = 0.38316

Guardado: /kaggle/working/xception_port/xception_param_weights.json


## 7. Guardar el modelo portado

In [9]:
torch.save({
    "state_dict": {k: v.cpu() for k, v in torch_model.state_dict().items()},
    "arch": "XceptionRegressor",
    "n_out": COND_DIM,
    "input_size": XPN_SIZE,
    "source_h5": XPN_WEIGHTS,
    "parity_max_abs_diff": float(diff.max()),
    "parity_ok": bool(PARITY_OK),
    "param_names": PARAM_NAMES,
    "r2_measured": r2s,
    "w": w.tolist(),
    "param_std": s_j.tolist(),
    "xpn_scaler_min": sc_xpn.data_min_.tolist(),
    "xpn_scaler_max": sc_xpn.data_max_.tolist(),
    "ddpm_scaler_min": sc_ddpm.data_min_.tolist(),
    "ddpm_scaler_max": sc_ddpm.data_max_.tolist(),
    "scalers_match": SCALERS_MATCH,
    "data_min": DATA_MIN, "data_max": DATA_MAX,
}, TORCH_CKPT)

print(f"Guardado: {TORCH_CKPT}  ({os.path.getsize(TORCH_CKPT)/1e6:.1f} MB)")
print("\n=== Resumen ===")
print(f"  Paridad Keras/Torch:  max dif {diff.max():.3e}  -> {'OK' if PARITY_OK else 'FALLIDA'}")
print(f"  Scalers equivalentes: {SCALERS_MATCH}")
print(f"  R2 medio (identificables T, Jex2, Kan1, KanS, Hex, KDM): "
      f"{np.mean([r2s[i] for i in [0,1,4,5,6,7]]):.3f}")
print("\nSiguiente paso: cuadernillo de fine-tuning por ciclo, que carga este .pt congelado.")

Guardado: /kaggle/working/xception_port/xception_regressor_torch.pt  (85.7 MB)

=== Resumen ===
  Paridad Keras/Torch:  max dif 6.974e-06  -> OK
  Scalers equivalentes: True
  R2 medio (identificables T, Jex2, Kan1, KanS, Hex, KDM): 0.834

Siguiente paso: cuadernillo de fine-tuning por ciclo, que carga este .pt congelado.


In [10]:
# Pegar como celda nueva al final de xception-keras-to-torch.ipynb.
# Separa dos causas posibles de un R2 bajo: mala calibracion afin (scaler
# equivocado) frente a perdida real de informacion.
#
# R2 no es invariante a reescalados afines de las predicciones, pero la
# correlacion si. Por eso corr^2 es el techo que alcanzaria R2 con la
# calibracion perfecta, y la brecha (corr^2 - R2) mide cuanto se pierde solo
# por escala y desplazamiento.

import numpy as np

print("  param      R2      corr^2   brecha    pendiente  intercepto")
for j, pn in enumerate(PARAM_NAMES):
    yt, yp = test_params[:, j], pred[:, j]
    r = np.corrcoef(yt, yp)[0, 1]
    r2 = r2_score(yt, yp)
    a, b = np.polyfit(yt, yp, 1)      # pred ~ a * true + b
    print(f"  {pn:8s} {r2:+7.3f} {r**2:8.3f} {r**2 - r2:8.3f}  {a:9.3f}  {b:+10.4f}")

print("\n  Lectura: pendiente ~1 e intercepto ~0 => el scaler es el correcto.")
print("  Pendiente lejos de 1 con corr^2 alto => el modelo extrae la senal pero")
print("  la conversion de escala esta mal (scaler de otro dataset).")

# Distribucion de los parametros: confirma las colas pesadas que reporta el paper
# para Kan1 y Hex, y explica por que su max es sensible al subconjunto usado.
print("\n  param        p50       p90       p99       max     max/p50")
for j, pn in enumerate(PARAM_NAMES):
    v = params_all[:, j]
    p50, p90, p99, mx = np.percentile(v, [50, 90, 99]).tolist() + [v.max()]
    ratio = mx / p50 if abs(p50) > 1e-9 else float("inf")
    print(f"  {pn:8s} {p50:9.4f} {p90:9.4f} {p99:9.4f} {mx:9.4f} {ratio:9.1f}")

# Escala recomendada para el termino de ciclo: std del parametro YA escalado a
# [0,1]. Evita el inverse_transform y no amplifica las colas como la std fisica.
rng_j = np.maximum(sc_xpn.data_max_ - sc_xpn.data_min_, 1e-12)
s_scaled = s_j / rng_j
print("\n  param     s_fisica   rango    s_escalada   w_j")
for j, pn in enumerate(PARAM_NAMES):
    print(f"  {pn:8s} {s_j[j]:9.4f} {rng_j[j]:8.4f} {s_scaled[j]:11.4f} {w[j]:7.4f}")

  param      R2      corr^2   brecha    pendiente  intercepto
  T         +0.946    0.946    0.000      0.929     +0.3029
  Jex2      +0.878    0.879    0.002      0.917     +0.0048
  Jex3      -0.083    0.025    0.107      0.076     -0.0008
  Jex4      -0.174    0.001    0.175      0.012     -0.0008
  Kan1      +0.617    0.625    0.007      0.692     +0.0321
  KanS      +0.670    0.672    0.002      0.707     +0.0213
  Hex       +0.902    0.903    0.000      0.896     +0.0056
  KDM       +0.988    0.989    0.000      0.976     +0.0125

  Lectura: pendiente ~1 e intercepto ~0 => el scaler es el correcto.
  Pendiente lejos de 1 con corr^2 alto => el modelo extrae la senal pero
  la conversion de escala esta mal (scaler de otro dataset).

  param        p50       p90       p99       max     max/p50
  T           3.4000    8.6000   13.7000   20.0000       5.9
  Jex2        0.0000    0.2190    0.6190    0.6590       inf
  Jex3        0.0000    0.0000    0.2560    0.2900       inf
  Jex4   